# GroundTruth — real multi-hazard build (clean notebook)

**Two real layers from free satellite data, rendered into one app:**
- **Urban heat** — Landsat thermal, 10-yr median + 2016→2025 time-lapse (national)
- **Ground deformation** — EGMS L3 InSAR, velocity + 2020→2024 time-lapse (per tile)

**Run order:** 1 (install → restart) → 2 (verify) → 3 (EE auth) → 4/5 (export heat tifs, *only once*) → 6 (mount) → 7 (upload EGMS CSV) → 8 (BUILD BOTH).

The heavy GEE exports (cells 4–5) only need running **once** — after the tifs are in Drive, skip straight to cell 8.


## 1 · Install egms-tools (then **Runtime → Restart session**)


In [5]:
# ===== EGMS-TOOLS INSTALL (reusable) =====
from google.colab import files
import glob, os
for f in glob.glob("egms-tools*.zip"): os.remove(f)
print("Upload the latest egms-tools.zip:")
up = files.upload()
!rm -rf egms-tools
!unzip -o -q egms-tools.zip
print("\n=== version in this zip ===")
!grep '^version' egms-tools/pyproject.toml
!pip uninstall -y egms-tools -q
!rm -rf /usr/local/lib/python3.12/dist-packages/egms_tools*
!pip install --no-cache-dir --force-reinstall --no-deps ./egms-tools -q
!pip install pyproj rasterio -q
print("\n" + "="*52)
print("STOP — Runtime → Restart session BEFORE the next cell.")
print("="*52)

Upload the latest egms-tools.zip:


Saving egms-tools.zip to egms-tools.zip

=== version in this zip ===
version = "0.41.0"
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

STOP — Runtime → Restart session BEFORE the next cell.


## 2 · Verify (after restart) — must print the version you installed

In [1]:
import egms_tools
print("version:", egms_tools.__version__)          # expect 0.23.0+
from egms_tools.io import load_egms_l3, egms_deformation_override
from egms_tools.surface import surface_png_stops, HEAT_STOPS, ANOM_STOPS
from egms_tools.heat import grid_from_array
print("loaders ready ")

version: 0.41.0
loaders ready 


## 3 · Earth Engine auth
Only needed if you will (re)export Landsat tifs. If your tifs are already in Drive, skip to cell 6.

In [2]:
import ee, geemap
ee.Authenticate()                       # run once
ee.Initialize(project="ee-sanjusajimon76")
print("EE initialized")

EE initialized


## 4 · Export 10-yr LST median to Drive  *(run once, then wait for Drive)*
Coarse 500 m to prove the chain. Watch the EE **Tasks** tab; the file lands in Drive `GroundTruth/`.

In [ ]:
import ee
def landsat_lst_to_drive(bbox, start, end, scale=500, max_cloud=60,
                         filename="germany_lst_10yr", folder="GroundTruth"):
    s, w, n, e = bbox
    region = ee.Geometry.Rectangle([w, s, e, n])
    def prep(img):
        qa = img.select("QA_PIXEL")
        cloud = qa.bitwiseAnd(1<<3).Or(qa.bitwiseAnd(1<<4))
        lst = img.select("ST_B10").multiply(0.00341802).add(149.0).subtract(273.15)
        return lst.updateMask(cloud.Not()).rename("LST").clip(region)
    col = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
           .merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))
           .filterBounds(region).filterDate(start, end)
           .filter(ee.Filter.lt("CLOUD_COVER", max_cloud))
           .filter(ee.Filter.calendarRange(6, 8, "month"))
           .map(prep))
    img = col.median().rename("LST")
    task = ee.batch.Export.image.toDrive(
        image=img, description=filename, folder=folder, fileNamePrefix=filename,
        region=region, scale=scale, maxPixels=1e10)
    task.start(); return task

task = landsat_lst_to_drive((47.3, 5.9, 55.1, 15.0), "2016-06-01", "2025-09-15",
                            scale=500, filename="germany_lst_10yr", folder="GroundTruth")
print("Export started:", task.status()["state"], "— watch the EE Tasks tab")

## 5 · Export 10 yearly LST tifs (2016–2025) *(run once)*

In [3]:
import ee
TODAY = "2026-06-27"   # current year is exported only up to today (partial summer)

def lst_year(year, scale=500):
    bbox = (47.3, 5.9, 55.1, 15.0); s, w, n, e = bbox
    region = ee.Geometry.Rectangle([w, s, e, n])
    end = f"{year}-08-31" if year < 2026 else TODAY    # 2026 = partial, up to today
    def prep(img):
        qa = img.select("QA_PIXEL")
        cloud = qa.bitwiseAnd(1<<3).Or(qa.bitwiseAnd(1<<4))
        lst = img.select("ST_B10").multiply(0.00341802).add(149.0).subtract(273.15)
        return lst.updateMask(cloud.Not()).rename("LST").clip(region)
    col = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
           .merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))
           .filterBounds(region).filterDate(f"{year}-06-01", end)
           .filter(ee.Filter.lt("CLOUD_COVER", 60)).map(prep))
    img = col.median().rename("LST")
    task = ee.batch.Export.image.toDrive(
        image=img, description=f"germany_lst_{year}", folder="GroundTruth",
        fileNamePrefix=f"germany_lst_{year}", region=region, scale=scale, maxPixels=1e10)
    task.start(); return task

# 2016-2025 = complete summers; 2026 = partial (the current heat wave)
#tasks = [lst_year(y) for y in range(2016, 2027)]
tasks = [lst_year(y) for y in [2026]]
print("started", len(tasks), "yearly exports (incl. partial 2026) — watch the Tasks tab")
print("2026 is exported only up to", TODAY, "= in-progress summer")

started 1 yearly exports (incl. partial 2026) — watch the Tasks tab
2026 is exported only up to 2026-06-27 = in-progress summer


## 6 · Mount Drive (where the LST tifs live)

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
print("GroundTruth files:", [f for f in os.listdir('/content/drive/MyDrive/GroundTruth') if f.endswith('.tif')][:12])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GroundTruth files: ['germany_lst_10yr.tif', 'germany_lst_2016.tif', 'germany_lst_2017.tif', 'germany_lst_2018.tif', 'germany_lst_2019.tif', 'germany_lst_2020.tif', 'germany_lst_2021.tif', 'germany_lst_2024.tif', 'germany_lst_2023.tif', 'germany_lst_2022.tif', 'germany_lst_2025.tif', 'germany_lst_2026.tif']


## 7 · Upload the EGMS deformation CSV
Download a tile from the Copernicus EGMS portal (ORTHO L3, Vertical). E45N31 = Lausitz (Germany).
Upload its CSV here, or point `CSV =` in cell 8 at a Drive path.

In [ ]:
# put all 4 _U_ CSVs in this folder (Drive, not /content)
TILES = "/content/drive/MyDrive/GroundTruth/egms_tiles"
from egms_tools.io import load_egms_l3_multi, egms_deformation_override
eg   = load_egms_l3_multi(TILES, max_points=400000)
defo = egms_deformation_override(eg, national=True)
print("tiles", eg["n_tiles"], "points", len(eg["vel"]), "filled", defo["filled_cells"])

## 8 · BUILD BOTH LAYERS → one app
This is the only cell you re-run when iterating. Builds heat + deformation overrides and calls `build_app` **once**, so you never lose a layer.

In [3]:
import rasterio, numpy as np, os
from matplotlib.path import Path
from egms_tools.germany import GERMANY_OUTLINE
from egms_tools.surface import surface_png_stops, HEAT_STOPS, ANOM_STOPS, fill_small_gaps
from egms_tools.heat import grid_from_array
from egms_tools.io import load_egms_l3_multi, egms_deformation_override
from egms_tools import build_app

folder = "/content/drive/MyDrive/GroundTruth"
poly   = Path(np.array(GERMANY_OUTLINE))
def clip_de(a, b):
    H, W = a.shape
    lons = np.linspace(b.left, b.right, W); lats = np.linspace(b.top, b.bottom, H)
    LON, LAT = np.meshgrid(lons, lats)
    a[~poly.contains_points(np.c_[LON.ravel(), LAT.ravel()]).reshape(H, W)] = np.nan
    a[(a < 5) | (a > 60)] = np.nan
    a = fill_small_gaps(a, max_gap_px=6)   # bridge thin Landsat scan seams
    return a

# ---- HEAT ----
with rasterio.open(f"{folder}/germany_lst_10yr.tif") as ds:
    lst = ds.read(1).astype("float32")
    if ds.nodata is not None: lst[lst == ds.nodata] = np.nan
    b = ds.bounds
lst = clip_de(lst, b)
baseline = float(np.nanmean(lst)); anomaly = lst - baseline
lo = float(np.nanpercentile(lst, 2)); hi = float(np.nanpercentile(lst, 98))
png_absolute = surface_png_stops(lst,     HEAT_STOPS, vmin=lo,  vmax=hi, alpha=0.9)
png_anomaly  = surface_png_stops(anomaly, ANOM_STOPS, vmin=-12, vmax=12, alpha=0.9)
heat_bounds  = [[b.bottom, b.left], [b.top, b.right]]
heat_grid    = grid_from_array(lst, heat_bounds, step=4)
frames, dates = [], []
for y in range(2016, 2026):
    p = f"{folder}/germany_lst_{y}.tif"
    if not os.path.exists(p): continue
    with rasterio.open(p) as ds:
        a = ds.read(1).astype("float32")
        if ds.nodata is not None: a[a == ds.nodata] = np.nan
        bb = ds.bounds
    a = clip_de(a, bb)
    frames.append(surface_png_stops(a, HEAT_STOPS, vmin=lo, vmax=hi, alpha=0.85))
    dates.append(str(y))
heat_override  = {"png": png_absolute, "png_anomaly": png_anomaly, "grid": heat_grid,
                  "bounds": heat_bounds, "baseline": round(baseline,1),
                  "vmin": round(lo,1), "vmax": round(hi,1)}
heat_timelapse = {"frames": frames, "dates": dates, "bounds": heat_bounds}
print("HEAT ready · baseline", round(baseline,1), "°C · timelapse", dates)

# ---- DEFORMATION ----
TILES = "/content/drive/MyDrive/GroundTruth/egms_tiles"
eg   = load_egms_l3_multi(TILES, max_points=800000)
defo = egms_deformation_override(eg, national=True, max_dist_deg=0.06)
print("DEFORMATION ready · tiles", eg["n_tiles"], "· points", len(eg["vel"]), "· filled", defo["filled_cells"])

# ---- BUILD ----
build_app("groundtruth_real.html", heat_override=heat_override,
          heat_timelapse=heat_timelapse, deformation_override=defo)
print("BUILT app")

HEAT ready · baseline 29.2 °C · timelapse ['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
DEFORMATION ready · tiles 5 · points 800000 · filled 23940
BUILT app


## 9 · PUBLISH Chapter 1 — Heat story + forecast
Run **right after cell 8, same session** (reuses `heat_override`, `heat_timelapse`, `clip_de`). Builds the scrollytelling heat story with the measured warming trend, an honest 2030 projection, and the partial-2026 frame if present.

In [4]:
# ===== CHAPTER 1: HEAT STORY + FORECAST + CITY GRAPHS + 2026-vs-NORMAL =====
import rasterio, numpy as np, os
from egms_tools.forecast import lst_warming_rate, lst_project, current_year_anomaly
from egms_tools.surface import surface_png_stops, HEAT_STOPS
from egms_tools.heat import grid_from_array
from egms_tools.germany import CITIES
from egms_tools import build_heat_story

folder = "/content/drive/MyDrive/GroundTruth"

year_arrays, year_list, year_grids = [], [], []
for y in range(2016, 2026):
    p = f"{folder}/germany_lst_{y}.tif"
    if not os.path.exists(p): continue
    with rasterio.open(p) as ds:
        a = ds.read(1).astype("float32")
        if ds.nodata is not None: a[a == ds.nodata] = np.nan
        bb = ds.bounds
    a = clip_de(a, bb)
    year_arrays.append(a); year_list.append(y)
    year_grids.append(grid_from_array(a, [[bb.bottom, bb.left],[bb.top, bb.right]], step=2))

def sample(grid, lat, lon):
    s,w,n,e = grid["bbox"]; nx,ny,z = grid["nx"],grid["ny"],grid["z"]
    if not (s<=lat<=n and w<=lon<=e): return None
    col=int((lon-w)/(e-w)*(nx-1)); row=int((n-lat)/(n-s)*(ny-1)); k=row*nx+col
    return z[k] if 0<=k<len(z) else None
city_series = {}
for name, lon, lat in CITIES:
    temps=[sample(g, lat, lon) for g in year_grids]
    if any(t is not None for t in temps): city_series[name]=(year_list, temps)

wr  = lst_warming_rate(year_arrays, year_list)
p30 = lst_project(year_arrays, year_list, 2030, baseline=heat_override["baseline"])
forecast = {"rate_png": wr["png"], "national_per_decade": wr["national_per_decade"],
            "projections": [p30], "bounds": heat_override["bounds"]}

current_year = None
partial_label = None
p26 = f"{folder}/germany_lst_2026.tif"
if os.path.exists(p26):
    with rasterio.open(p26) as ds:
        c26 = ds.read(1).astype("float32")
        if ds.nodata is not None: c26[c26 == ds.nodata] = np.nan
        bb = ds.bounds
    c26 = clip_de(c26, bb)
    heat_timelapse["frames"].append(surface_png_stops(c26, HEAT_STOPS,
        vmin=heat_override["vmin"], vmax=heat_override["vmax"], alpha=0.85))
    heat_timelapse["dates"].append("2026")
    partial_label = "2026 is a partial summer (in progress)"
    with rasterio.open(f"{folder}/germany_lst_10yr.tif") as ds:
        med = ds.read(1).astype("float32")
        if ds.nodata is not None: med[med == ds.nodata] = np.nan
        mb = ds.bounds
    med = clip_de(med, mb)
    grid_cur = grid_from_array(c26, heat_override["bounds"], step=2)
    grid_med = grid_from_array(med, heat_override["bounds"], step=2)
    current_year = current_year_anomaly(
        c26, med, heat_override["bounds"], 2026,
        cities=CITIES, grid_current=grid_cur, grid_median=grid_med,
        baseline_label="2016–2025",
        note="Summer 2026 measured through the latest cloud-free Landsat passes — still unfolding.")
    print(f"2026 vs normal: {current_year['national_delta']:+}°C national; "
          f"hottest-vs-normal {current_year['cities'][0]['name']} {current_year['cities'][0]['delta']:+}°C")

build_heat_story("heat_story.html",
    heat_override=heat_override, heat_timelapse=heat_timelapse,
    forecast=forecast, city_series=city_series,
    current_year=current_year, partial_label=partial_label,
    title="Germany is heating up", byline="Ground_truth")
print(f"Story built · {len(city_series)} cities with curves · warming {wr['national_per_decade']}°C/decade")
from google.colab import files
files.download("heat_story.html")

/usr/local/lib/python3.12/dist-packages/egms_tools/forecast.py:65: RuntimeWarning: invalid value encountered in divide
  sm = np.where(den > 0, num / den, np.nan)


2026 vs normal: +1.1°C national; hottest-vs-normal Dresden +4.7°C
Story built · 18 cities with curves · warming 1.4°C/decade


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>